# Shared network states and depicted emotion in *Friends* S3E15

This exploratory notebook fits seven shared network-specific PCA/HMM models to S3E15 only. MELD labels enter after decoding. Results are descriptive and in-sample; BOLD maps show relative supplied-signal patterns, not absolute activation.

In [1]:
from pathlib import Path
import sys
import warnings

import h5py
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from joblib import Parallel, delayed, parallel_config
from nilearn import image, plotting
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

working_path = Path.cwd().resolve()
PROJECT_ROOT = next((
    path for path in (working_path, *working_path.parents)
    if (path / 'pyproject.toml').is_file()
), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Could not find project root from {working_path}')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from preprocessing.brainstates.data import SegmentKey, discover_fmri_segments
from preprocessing.brainstates.networks import network_indices

SUBJECTS = ('sub-01', 'sub-02', 'sub-03', 'sub-05')
NETWORKS = ('VIS', 'SMN', 'DAN', 'VAN', 'LIM', 'FPC', 'DN')
TARGET_SEGMENTS = (SegmentKey(3, 15, 'a'), SegmentKey(3, 15, 'b'))
N_STATES = 4
N_COMPONENTS = 10
SEEDS = tuple(range(2025, 2030))
N_JOBS = 20
HMM_N_ITER = 500
HMM_TOL = 1e-3
MIN_STATE_OCCUPANCY = 0.01
MATCH_THRESHOLD = 0.90
PRIMARY_LAG = 3
TR_SECONDS = 1.49
LABEL_COLUMNS = ('meld_emotion', 'meld_sentiment')
LABEL_LEVELS = {
    'sentiment': ('negative', 'neutral', 'positive'),
    'emotion': ('neutral', 'anger', 'joy', 'sadness', 'surprise'),
}

FMRI_ROOT = PROJECT_ROOT / 'data/algonauts_2025.competitors/fmri'
LABEL_PATHS = {
    segment: PROJECT_ROOT / 'data/manual-data' / f'friends_{segment.task}.tsv'
    for segment in TARGET_SEGMENTS
}
RESULT_ROOT = PROJECT_ROOT / 'outputs/all_subjects_s03e15_network_emotion'
FIGURE_ROOT = RESULT_ROOT / 'figures'
MODEL_ROOT = RESULT_ROOT / 'models'
for path in (RESULT_ROOT, FIGURE_ROOT, MODEL_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_ROOT)
print('Episode:', [segment.task for segment in TARGET_SEGMENTS])
print('Subjects:', SUBJECTS)
print('Output:', RESULT_ROOT)

Project: /home/ducvu/project/Neuromatch-Algonauts2026
Episode: ['s03e15a', 's03e15b']
Subjects: ('sub-01', 'sub-02', 'sub-03', 'sub-05')
Output: /home/ducvu/project/Neuromatch-Algonauts2026/outputs/all_subjects_s03e15_network_emotion


In [2]:
def prepare_segment_labels(frame, target_length):
    """Filter uncertain MELD matches, pad, and shift within one segment."""
    if len(frame) > target_length:
        raise ValueError(f'{len(frame)} label rows for {target_length} TRs')
    missing = [column for column in (*LABEL_COLUMNS, 'match_ratio') if column not in frame]
    if missing:
        raise ValueError(f'Missing label columns: {missing}')

    prepared = frame.reindex(range(target_length)).copy()
    valid_match = prepared['match_ratio'].ge(MATCH_THRESHOLD)
    for column in LABEL_COLUMNS:
        prepared[column] = prepared[column].where(valid_match).shift(PRIMARY_LAG)
    prepared['accepted'] = prepared[list(LABEL_COLUMNS)].notna().all(axis=1)
    prepared.index.name = 'tr'
    return prepared


In [3]:
def pearson_r(values, labels):
    """Return Pearson r on finite, nonconstant paired rows."""
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels, dtype=float)
    keep = np.isfinite(values) & np.isfinite(labels)
    values = values[keep]
    labels = labels[keep]
    if len(values) < 2 or np.ptp(values) == 0 or np.ptp(labels) == 0:
        return np.nan
    return float(np.corrcoef(values, labels)[0, 1])


def correlation_rows(subject, network, gamma, labels):
    rows = []
    for label_type, levels in LABEL_LEVELS.items():
        column = f'meld_{label_type}'
        observed = labels[column].notna().to_numpy()
        observed_values = labels.loc[observed, column].to_numpy()
        for level in levels:
            indicator = (observed_values == level).astype(float)
            for state in range(gamma.shape[1]):
                rows.append({
                    'subject': subject, 'network': network, 'state': state,
                    'label_type': label_type, 'label': level,
                    'n_tr': int(observed.sum()),
                    'n_positive': int(indicator.sum()),
                    'pearson_r': pearson_r(gamma[observed, state], indicator),
                })
    return rows


In [4]:
def condition_mean_difference(values, labels, first, second):
    labels = np.asarray(labels)
    first_values = values[labels == first]
    second_values = values[labels == second]
    if not len(first_values) or not len(second_values):
        raise ValueError(f'Unsupported contrast: {first} versus {second}')
    return first_values.mean(axis=0) - second_values.mean(axis=0)


def back_project_state_means(state_means, pca, scalers):
    standardized = pca.inverse_transform(state_means)
    subject_maps = [
        scaler.inverse_transform(standardized)
        for scaler in scalers.values()
    ]
    return np.mean(subject_maps, axis=0)


def parcel_values_to_volume(parcel_values, atlas_labels):
    parcel_values = np.asarray(parcel_values, dtype=float)
    atlas_labels = np.asarray(atlas_labels, dtype=int)
    if parcel_values.shape != (1000,):
        raise ValueError(f'Expected 1,000 parcel values, got {parcel_values.shape}')
    if atlas_labels.min() < 0 or atlas_labels.max() > len(parcel_values):
        raise ValueError('Atlas labels fall outside parcel values')
    return np.concatenate(([0.0], parcel_values))[atlas_labels]


In [5]:
def build_network_dataset(fmri_by_sequence, parcel_indices):
    scalers = {}
    transformed = []
    keys = []
    lengths = []

    for subject in SUBJECTS:
        raw_parts = [
            fmri_by_sequence[(subject, segment)][:, parcel_indices]
            for segment in TARGET_SEGMENTS
        ]
        scalers[subject] = StandardScaler().fit(np.concatenate(raw_parts))
        for segment, raw in zip(TARGET_SEGMENTS, raw_parts):
            transformed.append(scalers[subject].transform(raw))
            keys.append((subject, segment))
            lengths.append(len(raw))

    pca = PCA(
        n_components=N_COMPONENTS, svd_solver='randomized', random_state=SEEDS[0],
    ).fit(np.concatenate(transformed))
    sequences = [pca.transform(values) for values in transformed]
    values = np.concatenate(sequences)
    if sum(lengths) != len(values):
        raise ValueError('Sequence lengths do not match concatenated values')
    return {
        'scalers': scalers, 'pca': pca, 'keys': keys,
        'lengths': lengths, 'values': values,
    }


def fit_hmm_restart(network, seed, values, lengths):
    model = GaussianHMM(
        n_components=N_STATES, covariance_type='full',
        n_iter=HMM_N_ITER, tol=HMM_TOL, random_state=seed,
    )
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=RuntimeWarning)
            model.fit(values, lengths)
        if not model.monitor_.converged:
            return None
        posterior = model.predict_proba(values, lengths)
        occupancy = posterior.mean(axis=0)
        if occupancy.min() < MIN_STATE_OCCUPANCY:
            return None
        score = float(model.score(values, lengths))
    except (ValueError, FloatingPointError, np.linalg.LinAlgError):
        return None
    if not np.isfinite(score):
        return None
    return network, model, score, occupancy, seed


def split_decoding(model, values, lengths, keys):
    posterior = model.predict_proba(values, lengths)
    states = model.predict(values, lengths)
    decoded = {}
    start = 0
    for key, length in zip(keys, lengths):
        stop = start + length
        decoded[key] = {
            'posterior': posterior[start:stop],
            'states': states[start:stop],
        }
        start = stop
    return decoded


## Load and audit S3E15

All four subjects contribute two independent 458-TR sequences.

In [6]:
fmri_by_sequence = {}
data_audit_rows = []
for subject in SUBJECTS:
    h5_path = (
        FMRI_ROOT / subject / 'func'
        / f'{subject}_task-friends_space-MNI152NLin2009cAsym_'
          'atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5'
    )
    if not h5_path.is_file():
        raise FileNotFoundError(h5_path)
    discovered = discover_fmri_segments(h5_path)
    with h5py.File(h5_path, 'r') as handle:
        for segment in TARGET_SEGMENTS:
            if segment not in discovered:
                raise ValueError(f'{subject}: missing {segment.task}')
            values = handle[discovered[segment]][:]
            if values.shape != (458, 1000) or not np.isfinite(values).all():
                raise ValueError(f'{subject}/{segment.task}: invalid {values.shape}')
            fmri_by_sequence[(subject, segment)] = values
            data_audit_rows.append({
                'subject': subject, 'segment': segment.task,
                'n_tr': values.shape[0], 'n_parcels': values.shape[1],
                'finite': True, 'h5_key': discovered[segment],
            })

data_audit = pd.DataFrame(data_audit_rows)
if len(data_audit) != 8:
    raise ValueError(f'Expected eight sequences, found {len(data_audit)}')
data_audit.to_csv(RESULT_ROOT / 'data_audit.csv', index=False)
data_audit


,subject,segment,n_tr,n_parcels,finite,h5_key
0,sub-01,s03e15a,458,1000,True,ses-024_task-s03e15a
1,sub-01,s03e15b,458,1000,True,ses-024_task-s03e15b
2,sub-02,s03e15a,458,1000,True,ses-023_task-s03e15a
3,sub-02,s03e15b,458,1000,True,ses-023_task-s03e15b
4,sub-03,s03e15a,458,1000,True,ses-034_task-s03e15a
5,sub-03,s03e15b,458,1000,True,ses-034_task-s03e15b
6,sub-05,s03e15a,458,1000,True,ses-016_task-s03e15a
7,sub-05,s03e15b,458,1000,True,ses-016_task-s03e15b


In [7]:
raw_labels = {}
labels_by_segment = {}
label_audit_rows = []
for segment, path in LABEL_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    raw = pd.read_csv(path, sep='\t')
    prepared = prepare_segment_labels(raw, target_length=458)
    raw_labels[segment] = raw
    labels_by_segment[segment] = prepared

    raw_labeled = raw[list(LABEL_COLUMNS)].notna().all(axis=1)
    high_score = raw['match_ratio'].ge(MATCH_THRESHOLD)
    for metric, count in (
        ('raw_labeled', raw_labeled.sum()),
        ('accepted_threshold', (raw_labeled & high_score).sum()),
        ('rejected_low_score', (raw_labeled & ~high_score).sum()),
        ('unlabeled', (~raw_labeled).sum()),
        ('padded_unlabeled', 458 - len(raw)),
    ):
        label_audit_rows.append({
            'segment': segment.task, 'metric': metric,
            'label_type': None, 'label': None, 'n_tr': int(count),
        })
    for label_type in ('emotion', 'sentiment'):
        column = f'meld_{label_type}'
        for label, count in prepared[column].value_counts().items():
            label_audit_rows.append({
                'segment': segment.task, 'metric': 'shifted_support',
                'label_type': label_type, 'label': label, 'n_tr': int(count),
            })

label_audit = pd.DataFrame(label_audit_rows)
accepted_count = int(label_audit.query("metric == 'accepted_threshold'")['n_tr'].sum())
if accepted_count != 186:
    raise ValueError(f'Expected 186 accepted labeled TRs, found {accepted_count}')
label_audit.to_csv(RESULT_ROOT / 'label_audit.csv', index=False)
label_audit


,segment,metric,label_type,label,n_tr
0,s03e15a,raw_labeled,None,None,126
1,s03e15a,accepted_threshold,None,None,106
2,s03e15a,rejected_low_score,None,None,20
3,s03e15a,unlabeled,None,None,331
4,s03e15a,padded_unlabeled,None,None,1
5,s03e15a,shifted_support,emotion,neutral,63
6,s03e15a,shifted_support,emotion,joy,17
7,s03e15a,shifted_support,emotion,surprise,10
8,s03e15a,shifted_support,emotion,anger,8
9,s03e15a,shifted_support,emotion,sadness,8


## Fit seven shared PCA/HMM models

Subject-specific parcel scaling removes subject offsets. Each network then shares one PCA and one four-state HMM across subjects, while the eight sequence lengths prevent artificial transitions.

In [8]:
parcel_indices = network_indices(PROJECT_ROOT / 'data/cache')
parcel_indices['VAN'] = parcel_indices.pop('SAL')
if sum(len(parcel_indices[network]) for network in NETWORKS) != 1000:
    raise ValueError('Network parcels do not cover the full atlas')

network_data = {
    network: build_network_dataset(fmri_by_sequence, parcel_indices[network])
    for network in NETWORKS
}
tasks = [
    delayed(fit_hmm_restart)(
        network, seed, network_data[network]['values'], network_data[network]['lengths']
    )
    for network in NETWORKS for seed in SEEDS
]
with parallel_config(
    backend='loky', n_jobs=min(N_JOBS, len(tasks)), inner_max_num_threads=1,
):
    fits = Parallel(verbose=10)(tasks)

artifacts = {}
hmm_summary_rows = []
for network in NETWORKS:
    valid = [fit for fit in fits if fit is not None and fit[0] == network]
    if not valid:
        raise RuntimeError(f'No healthy converged restart for {network}')
    _, model, training_ll, occupancy, seed = max(valid, key=lambda fit: fit[2])
    dataset = network_data[network]
    decoded = split_decoding(
        model, dataset['values'], dataset['lengths'], dataset['keys']
    )
    if any(
        not np.allclose(result['posterior'].sum(axis=1), 1.0)
        for result in decoded.values()
    ):
        raise ValueError(f'{network}: posterior rows do not sum to one')
    artifact = {
        'network': network, 'parcel_indices': parcel_indices[network],
        'scalers': dataset['scalers'], 'pca': dataset['pca'],
        'model': model, 'decoded': decoded, 'seed': seed,
        'training_log_likelihood': training_ll,
    }
    artifacts[network] = artifact
    joblib.dump(artifact, MODEL_ROOT / f'{network.lower()}_hmm.joblib')
    hmm_summary_rows.append({
        'network': network, 'n_states': N_STATES, 'n_components': N_COMPONENTS,
        'seed': seed, 'training_log_likelihood': training_ll,
        'min_state_occupancy': float(occupancy.min()),
        'iterations': model.monitor_.iter, 'converged': model.monitor_.converged,
    })

hmm_summary = pd.DataFrame(hmm_summary_rows)
hmm_summary.to_csv(RESULT_ROOT / 'hmm_summary.csv', index=False)
hmm_summary


[fetch_atlas_schaefer_2018] Dataset directory found: 
/home/ducvu/project/Neuromatch-Algonauts2026/data/cache/schaefer_2018

[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  4 out of 35 | elapsed:    1.2s remaining:    9.6s
[Parallel(n_jobs=20)]: Done  8 out of 35 | elapsed:    1.6s remaining:    5.5s
[Parallel(n_jobs=20)]: Done 12 out of 35 | elapsed:    1.9s remaining:    3.7s
[Parallel(n_jobs=20)]: Done 16 out of 35 | elapsed:    2.2s remaining:    2.6s
[Parallel(n_jobs=20)]: Done 20 out of 35 | elapsed:    2.3s remaining:    1.7s
[Parallel(n_jobs=20)]: Done 24 out of 35 | elapsed:    2.6s remaining:    1.2s
[Parallel(n_jobs=20)]: Done 28 out of 35 | elapsed:    2.8s remaining:    0.7s
[Parallel(n_jobs=20)]: Done 32 out of 35 | elapsed:    3.0s remaining:    0.3s
[Parallel(n_jobs=20)]: Done 35 out of 35 | elapsed:    3.3s finished


,network,n_states,n_components,seed,training_log_likelihood,min_state_occupancy,iterations,converged
0,VIS,4,10,2025,-83441.923744,0.138717,160,True
1,SMN,4,10,2028,-85510.683258,0.149611,93,True
2,DAN,4,10,2029,-78878.929882,0.138769,79,True
3,VAN,4,10,2027,-76061.606326,0.157442,167,True
4,LIM,4,10,2027,-66437.294324,0.123148,79,True
5,FPC,4,10,2029,-79911.868522,0.148068,163,True
6,DN,4,10,2026,-88151.886266,0.212114,128,True


## Pearson correlations with depicted MELD labels

Correlations are computed separately for every subject, network, state, and one-hot label. Missing time is excluded rather than treated as neutral.

In [9]:
posterior_frames = []
correlation_data = []
for subject in SUBJECTS:
    subject_labels = pd.concat(
        [labels_by_segment[segment].reset_index() for segment in TARGET_SEGMENTS],
        ignore_index=True,
    )
    for network in NETWORKS:
        decoded = artifacts[network]['decoded']
        gamma = np.concatenate([
            decoded[(subject, segment)]['posterior'] for segment in TARGET_SEGMENTS
        ])
        correlation_data.extend(
            correlation_rows(subject, network, gamma, subject_labels)
        )
        for segment in TARGET_SEGMENTS:
            result = decoded[(subject, segment)]
            labels = labels_by_segment[segment]
            for state in range(N_STATES):
                posterior_frames.append(pd.DataFrame({
                    'subject': subject, 'segment': segment.task,
                    'tr': np.arange(len(labels)), 'network': network,
                    'state': state, 'posterior': result['posterior'][:, state],
                    'viterbi_state': result['states'],
                    'meld_emotion': labels['meld_emotion'].to_numpy(),
                    'meld_sentiment': labels['meld_sentiment'].to_numpy(),
                }))

state_posteriors = pd.concat(posterior_frames, ignore_index=True)
pearson_correlations = pd.DataFrame(correlation_data)
if len(state_posteriors) != 102592 or len(pearson_correlations) != 896:
    raise ValueError(
        f'Unexpected output rows: {len(state_posteriors)}, {len(pearson_correlations)}'
    )
state_posteriors.to_csv(RESULT_ROOT / 'state_posteriors.csv', index=False)
pearson_correlations.to_csv(RESULT_ROOT / 'pearson_correlations.csv', index=False)
pearson_correlations.head()


,subject,network,state,label_type,label,n_tr,n_positive,pearson_r
0,sub-01,VIS,0,sentiment,negative,186,47,-0.175743
1,sub-01,VIS,1,sentiment,negative,186,47,0.003562
2,sub-01,VIS,2,sentiment,negative,186,47,0.120167
3,sub-01,VIS,3,sentiment,negative,186,47,-0.000573
4,sub-01,VIS,0,sentiment,neutral,186,101,0.133497


## Relative BOLD state signatures and label contrasts

HMM state means are back-projected to parcels and averaged equally across subject scalers. Direct contrasts are formed within subject before equal-weight averaging.

In [10]:
state_bold_rows = []
state_bold_vectors = {}
for network in NETWORKS:
    artifact = artifacts[network]
    projected = back_project_state_means(
        artifact['model'].means_, artifact['pca'], artifact['scalers']
    )
    for state in range(N_STATES):
        whole_brain = np.zeros(1000)
        whole_brain[artifact['parcel_indices']] = projected[state]
        state_bold_vectors[(network, state)] = whole_brain
        for parcel_index, value in zip(artifact['parcel_indices'], projected[state]):
            state_bold_rows.append({
                'network': network, 'state': state,
                'parcel_index': int(parcel_index), 'relative_bold': float(value),
            })

contrast_specs = {
    'sentiment_positive_minus_negative': ('meld_sentiment', 'positive', 'negative'),
    'emotion_anger_minus_neutral': ('meld_emotion', 'anger', 'neutral'),
    'emotion_joy_minus_neutral': ('meld_emotion', 'joy', 'neutral'),
    'emotion_sadness_minus_neutral': ('meld_emotion', 'sadness', 'neutral'),
    'emotion_surprise_minus_neutral': ('meld_emotion', 'surprise', 'neutral'),
}
label_bold_vectors = {}
label_bold_rows = []
for name, (column, first, second) in contrast_specs.items():
    subject_contrasts = []
    for subject in SUBJECTS:
        bold = np.concatenate([
            fmri_by_sequence[(subject, segment)] for segment in TARGET_SEGMENTS
        ])
        labels = pd.concat(
            [labels_by_segment[segment][column] for segment in TARGET_SEGMENTS],
            ignore_index=True,
        ).to_numpy()
        subject_contrasts.append(
            condition_mean_difference(bold, labels, first, second)
        )
    values = np.mean(subject_contrasts, axis=0)
    label_bold_vectors[name] = values
    for parcel_index, value in enumerate(values):
        label_bold_rows.append({
            'contrast': name, 'parcel_index': parcel_index,
            'relative_bold_difference': float(value),
        })

state_bold_maps = pd.DataFrame(state_bold_rows)
label_bold_contrasts = pd.DataFrame(label_bold_rows)
if len(state_bold_maps) != 4000 or len(label_bold_contrasts) != 5000:
    raise ValueError('Unexpected parcel-map row counts')
state_bold_maps.to_csv(RESULT_ROOT / 'state_bold_maps.csv', index=False)
label_bold_contrasts.to_csv(RESULT_ROOT / 'label_bold_contrasts.csv', index=False)
state_bold_maps.head()


,network,state,parcel_index,relative_bold
0,VIS,0,0,0.031284
1,VIS,0,1,0.051492
2,VIS,0,2,0.295002
3,VIS,0,3,0.277905
4,VIS,0,4,0.383895


## Figures

In [11]:
combined_labels = pd.concat(
    [labels_by_segment[segment] for segment in TARGET_SEGMENTS], ignore_index=True
)
support_rows = []
for label_type, levels in LABEL_LEVELS.items():
    counts = combined_labels[f'meld_{label_type}'].value_counts()
    for level in levels:
        support_rows.append({
            'label': f'{label_type}: {level}', 'n_tr': int(counts.get(level, 0))
        })
support = pd.DataFrame(support_rows)
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.bar(support['label'], support['n_tr'])
ax.set(title='S3E15 accepted MELD support after 3-TR shift', ylabel='TRs')
ax.tick_params(axis='x', rotation=35)
fig.savefig(FIGURE_ROOT / 'meld_label_support.png', dpi=180)
plt.close(fig)

sentiment_codes = {'negative': -1, 'neutral': 0, 'positive': 1}
emotion_codes = {label: index for index, label in enumerate(LABEL_LEVELS['emotion'])}
for subject in SUBJECTS:
    fig, axes = plt.subplots(
        len(NETWORKS) + 2, 1, figsize=(16, 15), sharex=True,
        gridspec_kw={'height_ratios': [1] * len(NETWORKS) + [0.35, 0.35]},
        constrained_layout=True,
    )
    for axis, network in zip(axes, NETWORKS):
        gamma = np.concatenate([
            artifacts[network]['decoded'][(subject, segment)]['posterior']
            for segment in TARGET_SEGMENTS
        ])
        axis.imshow(
            gamma.T, aspect='auto', origin='lower', interpolation='nearest',
            vmin=0, vmax=1, extent=[0, len(gamma), -0.5, N_STATES - 0.5],
        )
        axis.set(ylabel=network, yticks=range(N_STATES))
    sentiment = combined_labels['meld_sentiment'].map(sentiment_codes).to_numpy(float)
    emotion = combined_labels['meld_emotion'].map(emotion_codes).to_numpy(float)
    axes[-2].imshow(sentiment[None, :], aspect='auto', vmin=-1, vmax=1, cmap='coolwarm')
    axes[-2].set_ylabel('Sent.')
    axes[-2].set_yticks([])
    axes[-1].imshow(emotion[None, :], aspect='auto', vmin=0, vmax=4, cmap='tab10')
    axes[-1].set(ylabel='Emotion', xlabel='S3E15 TR')
    axes[-1].set_yticks([])
    for axis in axes:
        axis.axvline(458, color='white' if axis in axes[:-2] else 'black', linestyle='--')
    fig.suptitle(f'{subject}: shared network-state posteriors and depicted MELD labels')
    fig.savefig(FIGURE_ROOT / f'state_timeline_{subject}.png', dpi=180)
    plt.close(fig)

row_order = [(network, state) for network in NETWORKS for state in range(N_STATES)]
column_order = [
    (label_type, label)
    for label_type, levels in LABEL_LEVELS.items() for label in levels
]
fig, axes = plt.subplots(2, 2, figsize=(15, 18), constrained_layout=True)
for axis, subject in zip(axes.flat, SUBJECTS):
    subset = pearson_correlations.loc[pearson_correlations['subject'].eq(subject)].copy()
    subset['row'] = list(zip(subset['network'], subset['state']))
    subset['column'] = list(zip(subset['label_type'], subset['label']))
    matrix = subset.pivot(index='row', columns='column', values='pearson_r')
    matrix = matrix.reindex(index=row_order, columns=column_order)
    image_handle = axis.imshow(matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    axis.set_title(subject)
    axis.set_yticks(range(len(row_order)), [f'{n} S{s}' for n, s in row_order], fontsize=7)
    axis.set_xticks(
        range(len(column_order)), [f'{kind[:4]}:{label}' for kind, label in column_order],
        rotation=45, ha='right', fontsize=8,
    )
fig.colorbar(image_handle, ax=axes, label='Pearson r', shrink=0.5)
fig.suptitle('State posterior correlation with depicted MELD labels')
fig.savefig(FIGURE_ROOT / 'pearson_state_label_heatmaps.png', dpi=180)
plt.close(fig)


In [12]:
atlas_path = (
    PROJECT_ROOT / 'data/cache/schaefer_2018'
    / 'Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz'
)
if not atlas_path.is_file():
    raise FileNotFoundError(atlas_path)
atlas_img = image.load_img(atlas_path)
atlas_labels = np.asarray(atlas_img.dataobj, dtype=int)

def plot_parcel_map(values, axis, title):
    stat_map = image.new_img_like(
        atlas_img, parcel_values_to_volume(values, atlas_labels)
    )
    limit = float(np.nanmax(np.abs(values)))
    plotting.plot_stat_map(
        stat_map, axes=axis, display_mode='z', cut_coords=6,
        cmap='RdBu_r', symmetric_cbar=True, colorbar=True,
        threshold=1e-12, vmax=limit, title=title,
    )

for network in NETWORKS:
    fig, axes = plt.subplots(N_STATES, 1, figsize=(14, 12), constrained_layout=True)
    for state, axis in enumerate(axes):
        plot_parcel_map(
            state_bold_vectors[(network, state)], axis,
            f'{network} state {state}: PCA-back-projected relative BOLD',
        )
    fig.savefig(FIGURE_ROOT / f'state_bold_maps_{network.lower()}.png', dpi=150)
    plt.close(fig)

for name, values in label_bold_vectors.items():
    fig, axis = plt.subplots(figsize=(14, 4), constrained_layout=True)
    plot_parcel_map(values, axis, f"Relative BOLD: {name.replace('_', ' ')}")
    fig.savefig(FIGURE_ROOT / f'direct_bold_{name}.png', dpi=150)
    plt.close(fig)

expected_figures = 1 + len(SUBJECTS) + 1 + len(NETWORKS) + len(contrast_specs)
actual_figures = len(list(FIGURE_ROOT.glob('*.png')))
if actual_figures < expected_figures:
    raise ValueError(f'Expected at least {expected_figures} figures, found {actual_figures}')
print('Models:', len(list(MODEL_ROOT.glob('*.joblib'))))
print('Figures:', actual_figures)
print('Accepted labeled TRs:', accepted_count)


Models: 7
Figures: 18
Accepted labeled TRs: 186
